# TP3 : Dynamic programming for the knapsack problem


### Reading instances

In [2]:
function readKnaptxtInstance(filename)
    price=[]
    weight=[]
    KnapCap=[]
    open(filename) do f
        for i in 1:3
            tok = split(readline(f))
            if(tok[1] == "ListPrices=")
                for i in 2:(length(tok)-1)
                    push!(price,parse(Int64, tok[i]))
                end
            elseif(tok[1] == "ListWeights=")
                for i in 2:(length(tok)-1)
                    push!(weight,parse(Int64, tok[i]))
                end
            elseif(tok[1] == "Capacity=")
                push!(KnapCap, parse(Int64, tok[2]))
            else
                println("Unknown read :", tok)
            end 
        end
    end
    capacity=KnapCap[1]
    return price, weight, capacity
end

readKnaptxtInstance (generic function with 1 method)

### Dynamic programming : replace all TODO of the code

In [3]:
function solveKnapDP(filename)

    ## Loading data ##
    prices, weights, capacity_bag = readKnaptxtInstance(filename)

    ## Initialisation of the matrix ##
    n_items = length(prices)
    n_columns = capacity_bag + 1 # l'indexation dans julia commence par 1
    matrix_best_price = zeros(n_items, n_columns)

    ## Fulfilling th first row ##
    for capacity in (weights[1]+1):n_columns
        matrix_best_price[1,capacity] = prices[1]
    end
    
    ## Fulfilling the matrix with the recursive formula ##
    for item in 2:n_items
        for capacity in 1:n_columns
            if capacity >= weights[item]+1
                matrix_best_price[item, capacity] =  max(matrix_best_price[item-1, capacity], prices[item] + matrix_best_price[item-1, capacity - weights[item]])
            else
                # On ne prend pas l'objet, donc meme value précédent
                matrix_best_price[item, capacity] = matrix_best_price[item-1, capacity]
            end
        end
    end

    best_cost = matrix_best_price[n_items, n_columns]
    println("Best cost = ", best_cost)

    ## Backtracking to identify items selected
    remaining_capacity = capacity_bag + 1 
    list_items_selected = []

    ## Backtracking rows 2:n_items
    for item in n_items:-1:2
        # Si la valeur est différente de la colonne précédente => on a pris l'objet
        if matrix_best_price[item, remaining_capacity] != matrix_best_price[item-1, remaining_capacity]
            push!(list_items_selected, item)
            remaining_capacity -= weights[item]
        end
    end

    ## Checking whether item n°1 has been selected 
    if matrix_best_price[1,remaining_capacity] != 0
        push!(list_items_selected, 1)
    end

    println("items selected : ", reverse(list_items_selected))
    return best_cost, reverse(list_items_selected)
end

instance = "InstancesKnapSack/test.opb.txt"
println(solveKnapDP(instance))

Best cost = 65.0
items selected : Any[2, 4]
(65.0, Any[2, 4])


# Réponses aux questions

### La programmation dynamique et la relation de récurrence associée à un problème de programmation dynamique.
La programmation dynamique est une méthode algorithmique qui consiste à résoudre un problème complexe en le décomposant en sous-problèmes plus simples et en stockant les résultats pour ne pas les recalculer.
Pour le sac à dos, la relation de récurrence pour la valeur V(i, c) (objets 1..i, capacité c) est :
V(i, c) = max( V(i-1, c),  V(i-1, c - w_i) + p_i ) si l'objet rentre, sinon V(i-1, c).

### Adéquation du résultat
Le résultat obtenu (65 pour l'instance test) est cohérent avec l'instance fournie (on prend les objets 2 et 4 : 40 + 25 = 65, poids 4 + 5 = 9 < 10).

### Fonctionnement de l'algorithme implémenté
L'algorithme construit une matrice de taille (N, C+1) où N est le nombre d'objets et C la capacité du sac.
1. **Initialisation** : On remplit la première ligne pour le premier objet (on met sa valeur dans toutes les colonnes où la capacité est suffisante).
2. **Remplissage** : On parcourt la matrice ligne par ligne (objets) et colonne par colonne (capacités). Pour chaque case, on applique la relation de récurrence ci-dessus en regardant les valeurs de la ligne précédente.
3. **Backtracking** : Une fois la matrice remplie, la valeur optimale est en bas à droite (N, C). Pour trouver quels objets ont été pris, on remonte la matrice à l'envers : si la valeur V(i, c) est différente de V(i-1, c), c'est que l'objet i a été ajouté. On soustrait alors son poids à la capacité et on passe à l'objet précédent.


In [7]:
# Test sur plusieurs instances 
fichiers_test = [
    "InstancesKnapSack/Similar_Weights/KnapSack_100_1000_-995.opb.txt", # Similar_Weights
    "InstancesKnapSack/Strongly_Correlated/KnapSack_100_1000_-2397.opb.txt", # Strongly_Correlated
    "InstancesKnapSack/Uncorrelated/KnapSack_100_1000_-9147.opb.txt", # Uncorrelated
    "InstancesKnapSack/Weakly_Correlated/KnapSack_100_1000_-1514.opb.txt" # Weakly_Correlated
]

for instance in fichiers_test
    println("Instance : ", instance)
    t0 = time()
    solveKnapDP(instance)
    t1 = time() - t0
    println("Temps: ", t1)
    println()
end

Instance : InstancesKnapSack/Similar_Weights/KnapSack_100_1000_-995.opb.txt
Best cost = 995.0
items selected : Any[29]
Temps: 2.695348024368286

Instance : InstancesKnapSack/Strongly_Correlated/KnapSack_100_1000_-2397.opb.txt
Best cost = 2397.0
items selected : Any[2, 13, 21, 27, 30, 47, 51, 65, 71, 75, 77, 86, 90, 97]
Temps: 0.028415918350219727

Instance : InstancesKnapSack/Uncorrelated/KnapSack_100_1000_-9147.opb.txt
Best cost = 9147.0
items selected : Any[7, 11, 14, 24, 26, 31, 33, 38, 39, 49, 54, 61]
Temps: 0.02195596694946289

Instance : InstancesKnapSack/Weakly_Correlated/KnapSack_100_1000_-1514.opb.txt
Best cost = 1514.0
items selected : Any[11, 24, 33, 38, 45, 49, 57, 71, 85]
Temps: 0.021803855895996094



### Résultats de test

Voici les solutions obtenues et les valeurs de fonction-objectif sur les instances testées (toutes > 20 objets, sauf le test unitaire).

| Instance | Meilleur Coût (Obj) | Objets Sélectionnés | Temps (s) |
| :--- | :--- | :--- | :--- |
| **Similar_Weights** | **995.0** | `[29]` | 2.69534 |
| **Strongly_Correlated** | **2397.0** | `[2, 13, 21, 27, 30, 47, 51, 65, 71, 75, 77, 86, 90, 97]` | 0.02841 |
| **Uncorrelated** | **9147.0** | `[7, 11, 14, 24, 26, 31, 33, 38, 39, 49, 54, 61]` | 0.02195 |
| **Weakly_Correlated** | **1514.0** | `[11, 24, 33, 38, 45, 49, 57, 71, 85]` | 0.02180 |


### Comparaison avec le Branch-and-Bound

En comparant ces résultats avec ceux du Branch-and-Bound (TP2), nous observons des différences fondamentales de comportement :

**1. Le cas critique : "Strongly Correlated"**
* **Branch-and-Bound (Initial)** : **1510 s** (> 25 minutes). L'algorithme s'effondre car la relaxation linéaire est trop proche de la réalité mais non entière, empêchant l'élagage efficace de l'arbre.
* **Programmation Dynamique** : **0.028 s**.  
**->** La DP est insensible à la corrélation entre poids et valeurs. Sa complexité dépend uniquement du nombre d'objets et de la capacité du sac (O(N*C)). C'est la méthode à privilégier pour ces instances "pièges".

**2. La sensibilité à la capacité : "Similar Weights"**
* **Branch-and-Bound (Initial)** : **0.09 s**.
* **Programmation Dynamique** : **2.69 s**.  
**->** : Ici, la DP est **30 fois plus lente**. Cela s'explique probablement par une capacité du sac très élevée dans cette instance. Comme la DP doit remplir une matrice de taille N*C, elle devient lourde en mémoire et en temps lorsque C augmente, alors que le Branch-and-Bound est peu sensible à la valeur numérique de la capacité.

**3. Comparaison avec le Branch-and-Bound Optimisé**  

Il est intéressant de noter que votre version optimisée du Branch-and-Bound (0.002s sur Strongly) est plus rapide que la DP (0.028s). Cela montre qu'un Branch-and-Bound bien réglé (tri des objets + stratégie d'exploration) peut battre la DP, sauf si la capacité est très petite.


**Conclusion :**  
* La programmation dynamique est un algorithme robuste (temps stable, pas de "pire cas" imprévisible lié aux données) tant que la capacité du sac reste raisonnable.
* Le Branch-and-Bound est plus risqué (temps exponentiel possible sur Strongly Correlated) mais peut être extrêmement rapide sur des instances à grande capacité ou lorsqu'il est bien optimisé.

# BONUS : Algorithme de Bellman-Ford

In [4]:
# Structure pour représenter un arc (u -> v avec un coût)
struct Arc
    u::String
    v::String
    cout::Int
end

"""
Implémentation de l'algorithme de Bellman-Ford.
Retourne :
- distances : Dictionnaire {Sommet -> Coût minimal}
- predecesseurs : Dictionnaire {Sommet -> Sommet précédent dans le chemin optimal}
"""
function bellman_ford(sommets, arcs, source)
    # Initialisation  (Iteration 0)
    distances = Dict{String, Float64}()
    preds = Dict{String, Union{String, Nothing}}()
    # On met toutes les distances à l'infini (+Inf), sauf la source à 0
    for s in sommets
        distances[s] = Inf
        preds[s] = nothing
    end
    distances[source] = 0.0
    
    n = length(sommets)
    nb_iter = 0
    
    # Relâchement répété (jusqu'à n itérations ou stabilité)
    changement = true
    k = 1
    
    println("Début de l'algorithme (Source: $source)")
    
    while changement && k <= n
        changement = false
        print("Itération $k : ") # pour l'affichage de trace
        
        for arc in arcs
            if distances[arc.u] + arc.cout < distances[arc.v]
                distances[arc.v] = distances[arc.u] + arc.cout
                preds[arc.v] = arc.u
                changement = true
            end
        end
        
        # Affichage de l'état actuel pour vérifivation
        for s in sort(sommets)
            val = distances[s] == Inf ? "Inf" : Int(distances[s])
            print("$s=$val ")
        end
        println()
        
        k += 1
    end
    
    # Détection de cycle négatif (si k > n)
    if changement
        println("ATTENTION : Cycle de poids négatif détecté !")
        return nothing, nothing
    end
    
    return distances, preds
end

# Fonction pour reconstruire le chemin
function afficher_chemin(preds, cible)
    chemin = [cible]
    curr = cible
    while preds[curr] !== nothing
        curr = preds[curr]
        pushfirst!(chemin, curr)
    end
    return chemin
end

# Test avec l'exemple du sujet

# Définition du graphe basé sur le tableau de traces
noeuds_pdf = ["A", "B", "C", "D", "E", "F"]
arcs_pdf = [
    Arc("A", "B", 3), # Iter 1: B devient 3
    Arc("A", "E", 5), # Iter 1: E devient 5
    Arc("B", "C", 4), # Iter 2: C devient 3+4=7
    Arc("E", "D", 9), # Iter 2: D devient 5+9=14
    Arc("C", "D", 2), # Iter 3: D devient 7+2=9 
    Arc("D", "F", 3) # Iter 3: F devient 14+3=17, Iter 4: F devient 9+3=12
]

println("Exécution sur l'exemple du sujet")
println()
dists, parents = bellman_ford(noeuds_pdf, arcs_pdf, "A")

println()
println("Résultats finaux :")
println("Coûts minimaux : ", dists)

cible = "F"
if dists[cible] != Inf
    path = afficher_chemin(parents, cible)
    cout = Int(dists[cible])
    println("Plus court chemin vers $cible : ", join(path, " -> "))
    println("Coût total : Attendu = 12 | Obtenu = $cout ")
else
    println("Pas de chemin vers $cible")
end

Exécution sur l'exemple du sujet

Début de l'algorithme (Source: A)
Itération 1 : A=0 B=3 C=7 D=9 E=5 F=12 
Itération 2 : A=0 B=3 C=7 D=9 E=5 F=12 

Résultats finaux :
Coûts minimaux : Dict("B" => 3.0, "A" => 0.0, "C" => 7.0, "D" => 9.0, "E" => 5.0, "F" => 12.0)
Plus court chemin vers F : A -> B -> C -> D -> F
Coût total : Attendu = 12 | Obtenu = 12 
